In [5]:
import numpy as np
import pandas as pd
import rasterio
from rasterio.warp import calculate_default_transform, reproject, Resampling
from rasterio.windows import Window
from scipy.spatial import cKDTree
from pyproj import CRS
import os

In [6]:
# ================= SETTINGS =================
points_csv = "../../../Processed_Data/Processed_Lake.txt"          # columns: lon, lat
worldpop_tif = "global_pop_2015_CN_1km_R2025A_UA_v1_6933.tif"      # population per cell (people)

MAX_KM = 105      # how far out to count population
BIN_KM = 5      # histogram bin size

# The code will automatically convert everything to meters
TARGET_CRS = CRS.from_epsg(6933)   # global, meters, safe for population

In [7]:
# ================= LOAD POINTS =================
pts = pd.read_csv(points_csv)

# keep only points with THE SEPCIFIC LAKE TYPE
lake_type = "Natural lake"
pts = pts[pts['TypeName'] == lake_type]
pts_xy_lonlat = pts[["X", "Y"]].values

# ================= MAKE RASTER METRIC =================
def make_metric_raster(src_path):
    with rasterio.open(src_path) as src:
        if src.crs is None:
            raise ValueError("WorldPop file has no map info (CRS).")

        if CRS(src.crs) == TARGET_CRS:
            return src_path

        out = src_path.replace(".tif", "_meters.tif")

        transform, width, height = calculate_default_transform(
            src.crs, TARGET_CRS, src.width, src.height, *src.bounds
        )

        profile = src.profile
        profile.update({
            "crs": TARGET_CRS,
            "transform": transform,
            "width": width,
            "height": height
        })

        with rasterio.open(out, "w", **profile) as dst:
            for i in range(1, src.count + 1):
                reproject(
                    rasterio.band(src, i),
                    rasterio.band(dst, i),
                    src_transform=src.transform,
                    src_crs=src.crs,
                    dst_transform=transform,
                    dst_crs=TARGET_CRS,
                    resampling=Resampling.nearest  # do NOT smooth population
                )

    return out

raster_m = make_metric_raster(worldpop_tif)

# ================= PROJECT POINTS =================
from pyproj import Transformer
transformer = Transformer.from_crs("EPSG:4326", TARGET_CRS, always_xy=True)
pts_x, pts_y = transformer.transform(
    pts_xy_lonlat[:,0], pts_xy_lonlat[:,1]
)
tree = cKDTree(np.column_stack([pts_x, pts_y]))

# ================= HISTOGRAM SETUP =================
bins_m = np.arange(0, (MAX_KM + BIN_KM) * 1000, BIN_KM * 1000)
hist = np.zeros(len(bins_m) - 1)

# ================= PROCESS WORLDPOP =================
with rasterio.open(raster_m) as src:
    for row in range(0, src.height, 1024):
        for col in range(0, src.width, 1024):

            window = Window(col, row, 1024, 1024)
            pop = src.read(1, window=window, masked=True)

            if pop.mask.all():
                continue

            rows, cols = np.nonzero((~pop.mask) & (pop > 0))
            if len(rows) == 0:
                continue

            rows += row
            cols += col

            xs, ys = rasterio.transform.xy(src.transform, rows, cols, offset="center")
            xs = np.array(xs)
            ys = np.array(ys)
            vals = pop.data[~pop.mask & (pop > 0)]

            dists, _ = tree.query(np.column_stack([xs, ys]), k=1)

            use = dists <= MAX_KM * 1000
            dists = dists[use]
            vals = vals[use]

            idx = np.searchsorted(bins_m, dists, side="right") - 1
            ok = (idx >= 0) & (idx < len(hist))
            np.add.at(hist, idx[ok], vals[ok])

# ================= SAVE RESULT =================
out = pd.DataFrame({
    "distance_km": bins_m[:-1] / 1000,
    "population": hist
})

out.to_csv(f"../{lake_type}_population_histogram_max_{MAX_KM}km_bin_{BIN_KM}km.csv", index=False)
print(out.head())
print(f"Done. Saved.")

   distance_km    population
0          0.0  4.798717e+05
1          5.0  5.471833e+06
2         10.0  1.433547e+07
3         15.0  2.352887e+07
4         20.0  3.100347e+07
Done. Saved.
